# TOTNet Football Training - Google Colab

This notebook trains TOTNet on your custom football dataset using Google Colab's free GPU.

## Overview
1. Setup environment and check GPU
2. Upload/prepare data
3. Train TOTNet (fine-tune from Tennis weights)
4. Evaluate and save model

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

# Should show Tesla T4 (free tier) or better

In [ ]:
# Install dependencies
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q easydict matplotlib opencv-python scikit-learn scipy tqdm tensorboard ptflops

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Option A: Clone TOTNet from GitHub (replace with your repo URL)
# !git clone https://github.com/your-username/TOTNet.git /content/TOTNet
# %cd /content/TOTNet

# Option B: Upload TOTNet code from Drive (faster if already saved)
# Assuming you've zipped TOTNet and uploaded to Drive
!unzip -q /content/drive/MyDrive/TOTNet.zip -d /content/
%cd /content/TOTNet

## 2. Prepare Data

In [ ]:
# Option A: Upload prepared football_dataset from Drive
# Assumes you ran prepare_football_data.py locally and zipped the output
!unzip -q /content/drive/MyDrive/football_dataset.zip -d /content/TOTNet/data/

# Verify data structure
!ls -la /content/TOTNet/data/football_dataset/
!echo "Frame directories:"
!ls /content/TOTNet/data/football_dataset/frames/

In [ ]:
# Option B: Run data preparation script (if you uploaded raw augmented sequences)
# This is slower but works if you uploaded ball-tracking-labeler output directly
# !python src/data_process/prepare_football_data.py \
#     --source_dir /content/drive/MyDrive/ball-tracking-labeler \
#     --output_dir /content/TOTNet/data/football_dataset \
#     --resize 512 288 \
#     --test_ratio 0.2 \
#     --seed 42

In [ ]:
# Verify dataset statistics
import json

with open('/content/TOTNet/data/football_dataset/train.json') as f:
    train_data = json.load(f)
with open('/content/TOTNet/data/football_dataset/test.json') as f:
    test_data = json.load(f)

print(f"Train videos: {len(train_data)}")
print(f"Test videos: {len(test_data)}")

total_train_frames = sum(len(v['ball_pos']) for v in train_data)
total_test_frames = sum(len(v['ball_pos']) for v in test_data)
print(f"Total train frames: {total_train_frames}")
print(f"Total test frames: {total_test_frames}")

## 3. Training

In [ ]:
# Create directories
!mkdir -p /content/TOTNet/checkpoints/TOTNet_Football
!mkdir -p /content/TOTNet/logs/TOTNet_Football

In [ ]:
# Check for pretrained Tennis weights
!ls -la /content/TOTNet/weights/

In [ ]:
# Start TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/TOTNet/logs/

In [ ]:
# Training command - Single GPU
# Adjust batch_size based on GPU memory (T4: 8-12, A100: 16-24)

!python src/main.py \
    --num_epochs 30 \
    --saved_fn 'TOTNet_Football' \
    --num_frames 5 \
    --optimizer_type adamw \
    --lr 5e-4 \
    --loss_function WBCE \
    --weight_decay 5e-5 \
    --img_size 288 512 \
    --batch_size 12 \
    --print_freq 50 \
    --dataset_choice 'football' \
    --model_choice 'TOTNet' \
    --weighting_list 1 2 2 3 \
    --occluded_prob 0.1 \
    --ball_size 4 \
    --val-size 0.2 \
    --pretrained_path 'weights/TOTNet_Tennis_(5)_(288,512)_30epochs_Occl(0.25)_WBCE[1,2,3,3]_bs8_ch64/TOTNet_Tennis_(5)_(288,512)_30epochs_Occl(0.25)_WBCE[1,2,3,3]_bs8_ch64_best.pth' \
    --no_test

## 4. Evaluation & Export

In [ ]:
# List trained model checkpoints
!ls -la /content/TOTNet/checkpoints/TOTNet_Football/

In [ ]:
# Copy best model to Google Drive
!cp /content/TOTNet/checkpoints/TOTNet_Football/TOTNet_Football_best.pth \
    /content/drive/MyDrive/TOTNet_Football_best.pth

print("Model saved to Google Drive!")

In [ ]:
# Optional: Run evaluation on test set
# !python src/test.py \
#     --pretrained_path 'checkpoints/TOTNet_Football/TOTNet_Football_best.pth' \
#     --dataset_choice 'football' \
#     --model_choice 'TOTNet' \
#     --img_size 288 512

## 5. (Optional) Run Demo

In [ ]:
# Demo on a test video
# Upload a test video to Drive first

# !python src/demo.py \
#     --video_path '/content/drive/MyDrive/test_football.mp4' \
#     --pretrained_path 'checkpoints/TOTNet_Football/TOTNet_Football_best.pth' \
#     --model_choice 'TOTNet' \
#     --img_size 288 512 \
#     --output_format video \
#     --save_demo_output